# Same-genotype comparison demo

Compares multiple experiments that share a genotype: prints each experiment's genotype
up front (cross-checked against the h5, so a typo/stale label is caught immediately),
runs the same baseline QC as `2_data_quality_demo.ipynb` for each one individually, then
pools cells across all of them for a population-level contrast-response comparison via
`ra.plot_crf_across_experiments`.

Genotype isn't in the DataJoint schema yet -- `ra.get_genotype(exp_name)` reads it
straight from each h5's source properties. Most experiments don't have a genotype
recorded yet (see the genotype audit in `changes/`); this notebook only works for
experiments where it's been filled in.

In [ ]:
import retinanalysis as ra
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
%matplotlib inline

## Experiments to compare

Set `EXP_NAMES` to the experiments you want to pool, and `GENOTYPE` to what you expect
them all to be -- the next cell checks every experiment's actual h5 genotype against
this and flags any mismatch or missing value before you go any further.

In [ ]:
EXP_NAMES = [
    # e.g. '20260715A', '20260716A',
]
GENOTYPE = 'rd10'  # expected genotype shared across EXP_NAMES

In [ ]:
genotype_by_exp = {}
for exp_name in EXP_NAMES:
    found = ra.get_genotype(exp_name)
    genotype_by_exp[exp_name] = found
    flag = '' if found == GENOTYPE else '  <-- MISMATCH or missing'
    print(f'{exp_name}: {found!r}{flag}')

mismatches = [e for e, g in genotype_by_exp.items() if g != GENOTYPE]
if mismatches:
    print(f'\n{len(mismatches)} experiment(s) do not match GENOTYPE ({GENOTYPE!r}): {mismatches}')
    print('Fix EXP_NAMES/GENOTYPE or the h5 genotype field before trusting anything below.')
else:
    print(f'\nAll {len(EXP_NAMES)} experiments confirmed as {GENOTYPE!r}.')

## Baseline sanity checks (per experiment)

Same QC as `2_data_quality_demo.ipynb`, run once per experiment in `EXP_NAMES`: pick
each experiment's classified white-noise chunk, build a pipeline, and look at spike
counts and RF mosaics. Skim these before trusting the pooled contrast comparison below
-- a bad experiment pulled into the pooled CRF is hard to spot after the fact.

In [ ]:
noise_search = ra.get_datasets_from_protocol_names('spatialnoise')

pipelines_by_exp = {}
for exp_name in EXP_NAMES:
    print(f'=== {exp_name} ===')
    datafile_name = ra.find_datafile_for_protocol(noise_search, exp_name)
    analysis_chunk_name = ra.find_classified_noise_chunk(exp_name)
    if datafile_name is None or analysis_chunk_name is None:
        print(f'{exp_name}: missing a datafile or classified chunk, skipping QC for this experiment.')
        continue
    with ra.scrollable_prints():
        pipeline = ra.create_mea_pipeline(exp_name, datafile_name, analysis_chunk_name=analysis_chunk_name)
    pipelines_by_exp[exp_name] = pipeline

In [ ]:
for exp_name, pipeline in pipelines_by_exp.items():
    print(f'=== {exp_name}: spike counts ===')
    spike_times = ra.get_spike_xarr(pipeline.resp, minimum_n=3)
    spike_counts = xr.apply_ufunc(len, spike_times, vectorize=True)
    print(spike_counts.mean(dim=['epoch']).to_pandas())

In [ ]:
for exp_name, pipeline in pipelines_by_exp.items():
    pipeline.plot_rfs(minimum_n=3, b_zoom=True, title=f'{exp_name} ({GENOTYPE}) -- RFs by Cell Type')

## Contrast comparison, pooled across experiments

Pools cells from every experiment in `EXP_NAMES` into one averaged CRF per NDF per cell
type, via `ra.plot_crf_across_experiments` -- see its docstring for how per-experiment
chunk auto-detection and cell-id collisions across experiments are handled.

In [ ]:
contrast_search = ra.get_datasets_from_protocol_names('contrast')
GRATING_PROTOCOL_NAME = 'manookinlab.protocols.ContrastResponseGrating'
CORR_THRESHOLD = 0.8

In [ ]:
figs_crf, df_crf_points = ra.plot_crf_across_experiments(
    EXP_NAMES, contrast_search, GRATING_PROTOCOL_NAME, condition_key='contrast',
    corr_cutoff=CORR_THRESHOLD, response_col='f1',
    baseline_condition_key='contrast', baseline_condition_value=0.0,
    title_prefix=f'{GENOTYPE}: F1 vs. contrast across NDFs, pooled across experiments',
    even_spacing=True,
)
ra.scrollable_dataframe(df_crf_points)